# 04 · Modelling — Favorita

Modellvergleich auf dem gleichen chronologischen Split:

| Gruppe | Modell | Datenformat |
|---|---|---|
| Statistisch | SARIMAX | 1D-Serie pro Store |
| Statistisch | Prophet | 1D-Serie + exogene Regressoren |
| ML | XGBoost | 2D-Feature-Matrix |
| ML | LightGBM | 2D-Feature-Matrix |
| Deep Learning | PatchTST | Sequenzen via neuralforecast |
| Deep Learning | NHITS | Sequenzen + stat/hist exog via neuralforecast |

## 0 · Imports & Setup

In [2]:
import os
import sys
import pathlib
import tempfile
from pathlib import Path

os.environ['CMDSTANPY_NOT_USE_POLARS'] = 'True'  # muss vor prophet-Import stehen

sys.path.append(os.path.abspath('../03_src'))

import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utilis import run_prophet, run_sarimax, evaluate, run_xgb, run_lgbm, to_nf_format
from config import (FINAL, TARGET_COL, FEATURE_COLS,EXOG_COLS, HIST_EXOG, STAT_EXOG,TRAIN_END, VAL_END, LOOKBACK, HORIZON)

RESULTS = FINAL / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')

ImportError: cannot import name 'EXOG_COLS' from 'config' (c:\Users\maxkr\E-Commerce-Time-Series-Forecasting\03_src\config.py)

## 1 · Daten laden & Split

In [ ]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train = df.filter(pl.col('date') <= TRAIN_END)
val   = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test  = df.filter(pl.col('date') > VAL_END)

print(f'Train: {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:   {val.shape}    {val["date"].min()}   → {val["date"].max()}')
print(f'Test:  {test.shape}   {test["date"].min()}  → {test["date"].max()}')

stores = train['store_nbr'].unique().sort().to_list()
print(f'Stores: {len(stores)}')

In [ ]:
# Format 1: ML-Feature-Matrizen (für XGBoost & LightGBM)
X_train = train.select(FEATURE_COLS).to_numpy()
y_train = train.select(TARGET_COL).to_numpy().ravel()

X_val   = val.select(FEATURE_COLS).to_numpy()
y_val   = val.select(TARGET_COL).to_numpy().ravel()

X_test  = test.select(FEATURE_COLS).to_numpy()
y_test  = test.select(TARGET_COL).to_numpy().ravel()

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
train_nf = to_nf_format(train)
val_nf   = to_nf_format(val)

# static_df: ein Eintrag pro Store (für NHITS)
static_df = (
    train.select(['store_nbr'] + STAT_EXOG)
         .unique(subset=['store_nbr'])
         .sort('store_nbr')
         .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
         .select(['unique_id'] + STAT_EXOG)
         .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
         .to_pandas()
)

print('NeuralForecast Format:')
print(train_nf.dtypes)
print(f'static_df shape: {static_df.shape}')
train_nf.head(3)

## 2 · Statistisch: SARIMAX

Läuft pro Store separat. Exogene Regressoren: Ölpreis + Feiertags-Flags.  
Saisonalität: `seasonal_order=(1,1,1,7)` → wöchentlich.

In [ ]:
preds_sarimax = run_sarimax(
    train, val,
    stores=stores,
    extra_regressors=EXOG_COLS
)
preds_sarimax.write_parquet(RESULTS / 'val_sarimax.parquet')
preds_sarimax.head(5)

## 3 · Statistisch: Prophet

Läuft pro Store separat. Exogene Regressoren: Ölpreis.

In [ ]:
preds_prophet = run_prophet(
    train, val,
    stores=stores,
    extra_regressors=['oil_price']
)
preds_prophet.write_parquet(RESULTS / 'val_prophet.parquet')
preds_prophet.head(5)

## 4 · ML: XGBoost & LightGBM

Globale Modelle auf der gesamten Feature-Matrix. Kein Store-Loop nötig.

In [ ]:
preds_xgb = run_xgb(X_train, y_train, X_val, y_val, train, val)
preds_xgb.write_parquet(RESULTS / 'val_xgb.parquet')
preds_xgb.head(5)

In [ ]:
preds_lgbm = run_lgbm(X_train, y_train, X_val, y_val, train, val)
preds_lgbm.write_parquet(RESULTS / 'val_lgbm.parquet')
preds_lgbm.head(5)

## 5 · Deep Learning: PatchTST

Transformer-basiertes Modell auf Patch-Ebene.  
Keine exogenen Features (PatchTST unterstützt keine stat/hist exog).

In [ ]:
import torch
torch.set_num_threads(1)

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

model_patchtst = PatchTST(
    h=HORIZON,
    input_size=LOOKBACK,
    patch_len=16,
    stride=8,
    encoder_layers=2,
    n_heads=8,
    hidden_size=64,
    linear_hidden_size=128,
    dropout=0.2,
    fc_dropout=0.2,
    scaler_type='standard',
    max_steps=200,
    batch_size=64,
    learning_rate=1e-4,
    early_stop_patience_steps=10,
    val_check_steps=25,
    random_seed=42,
)

nf_patchtst = NeuralForecast(models=[model_patchtst], freq='D')
nf_patchtst.fit(
    df=train_nf[['unique_id', 'ds', 'y']],
    val_size=len(val['date'].unique())
)

preds_patchtst = nf_patchtst.predict()

preds_patchtst_pl = pl.from_pandas(preds_patchtst.reset_index()).rename({
    'unique_id': 'store_nbr',
    'ds': 'date',
    'PatchTST': 'pred_patchtst',
}).with_columns([
    pl.col('store_nbr').cast(pl.Int64),
    pl.col('date').cast(pl.Date),
])

preds_patchtst_pl.write_parquet(RESULTS / 'val_patchtst.parquet')
preds_patchtst_pl.head(5)

## 6 · Deep Learning: NHITS

Hierarchisches Interpolations-Modell.  
Unterstützt statische (`store_type_enc`, `cluster`) und historische exogene Features (`oil_price`, Feiertage).  
Deutlich schneller als TFT auf CPU bei vergleichbarer Qualität.

In [ ]:
from neuralforecast.models import NHITS

required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG

model_nhits = NHITS(
    h=HORIZON,
    input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG,
    hist_exog_list=HIST_EXOG,
    scaler_type='standard',
    max_steps=200,
    batch_size=128,
    learning_rate=1e-3,
    early_stop_patience_steps=5,
    val_check_steps=10,
    random_seed=42,
)

nf_nhits = NeuralForecast(models=[model_nhits], freq='D')
nf_nhits.fit(
    df=train_nf[required_cols_nhits],
    static_df=static_df,
    val_size=len(val['date'].unique())
)

preds_nhits = nf_nhits.predict()

preds_nhits_pl = pl.from_pandas(preds_nhits.reset_index()).rename({
    'unique_id': 'store_nbr',
    'ds': 'date',
    'NHITS': 'pred_nhits',
}).with_columns([
    pl.col('store_nbr').cast(pl.Int64),
    pl.col('date').cast(pl.Date),
])

preds_nhits_pl.write_parquet(RESULTS / 'val_nhits.parquet')
preds_nhits_pl.head(5)

## 7 · Evaluation & Vergleich

Alle gespeicherten Vorhersagen laden und gemeinsam auswerten.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(df, pred_col, label):
    merged = val.join(
        df.select(['store_nbr', 'date', pred_col]),
        on=['store_nbr', 'date'],
        how='inner'
    ).drop_nulls()
    y_true = merged[TARGET_COL].to_numpy()
    y_pred = merged[pred_col].to_numpy()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    print(f'{label:<20} MAE={mae:.1f}  RMSE={rmse:.1f}  MAPE={mape:.1f}%')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []

for fname, pred_col, label in [
    ('val_sarimax.parquet',  'pred_sarimax',  'SARIMAX'),
    ('val_prophet.parquet',  'pred_prophet',  'Prophet'),
    ('val_xgb.parquet',      'pred_xgb',      'XGBoost'),
    ('val_lgbm.parquet',     'pred_lgbm',     'LightGBM'),
    ('val_patchtst.parquet', 'pred_patchtst', 'PatchTST'),
    ('val_nhits.parquet',    'pred_nhits',    'NHITS'),
]:
    try:
        preds = pl.read_parquet(RESULTS / fname)
        results.append(compute_metrics(preds, pred_col, label))
    except FileNotFoundError:
        print(f'{label:<20} ← Datei nicht gefunden, übersprungen')

results_df = pd.DataFrame(results).set_index('label').sort_values('MAE')
print('\n=== Ranking nach MAE ===')
print(results_df.round(2))

In [ ]:
# Visualisierung: Metriken-Vergleich
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    results_df[metric].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(metric)
    ax.set_xlabel(metric)

plt.suptitle('Modellvergleich — Validierungsset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisierung: Forecast vs Ist-Wert für Store 1
sample_store = 1
val_store = val.filter(pl.col('store_nbr') == sample_store).sort('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(val_store['date'].to_list(), val_store[TARGET_COL].to_list(),
        label='Ist-Wert', color='black', linewidth=2)

colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']
for (fname, pred_col, label), color in zip([
    ('val_sarimax.parquet',  'pred_sarimax',  'SARIMAX'),
    ('val_prophet.parquet',  'pred_prophet',  'Prophet'),
    ('val_xgb.parquet',      'pred_xgb',      'XGBoost'),
    ('val_lgbm.parquet',     'pred_lgbm',     'LightGBM'),
    ('val_patchtst.parquet', 'pred_patchtst', 'PatchTST'),
    ('val_nhits.parquet',    'pred_nhits',    'NHITS'),
], colors):
    try:
        preds = pl.read_parquet(RESULTS / fname)
        store_preds = preds.filter(pl.col('store_nbr') == sample_store).sort('date')
        ax.plot(store_preds['date'].to_list(), store_preds[pred_col].to_list(),
                label=label, linestyle='--', color=color, alpha=0.8)
    except FileNotFoundError:
        pass

ax.set_title(f'Forecast vs Ist-Wert — Store {sample_store}', fontsize=13)
ax.legend()
ax.set_xlabel('Datum')
ax.set_ylabel(TARGET_COL)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RESULTS / f'forecast_store{sample_store}.png', dpi=150, bbox_inches='tight')
plt.show()